In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("prepared_positive_v81.csv")
print(f"Loaded: {df.shape}")
print(f"Conversion rate: {df['converted'].mean()*100:.2f}%")
 
 
CLIENT_FEATURES = [
    # Numeric — use scaled versions
    "quote_value_scaled",
    "lead_difficulty_scaled",
    "sophistication_scaled",
    "patience_hours_scaled",
    "digital_engagement_score_scaled",
    "tenure_years_scaled",
    "log_quote_value",
    "log_patience_hours",
    # Temporal
    "month",
    "hour_of_day",
    "lead_dayofweek",
    "lead_quarter",
    "is_weekend",
    # Encoded categoricals
    "insurance_type_enc",
    "claims_risk",
    "multi_product_intent",
    # Missing flags (informative)
    "insurance_type_missing",
    "language_missing",
    "tenure_years_missing",
    "digital_engagement_score_missing",
]
 
# ── Broker features ───────────────────────────────────────
BROKER_FEATURES = [
    # Numeric — use scaled versions
    "skill_level_scaled",
    "conversion_rate_scaled",
    "csat_score_scaled",
    "reliability_scaled",
    "efficiency_scaled",
    "avg_response_time_scaled",
    "burnout_risk_scaled",
    "commission_rate_scaled",
    "cost_per_lead_scaled",
    "utilization_scaled",
    # Raw useful numerics
    "ribo_experience",
    "ribo_licensed",
    "is_new_broker",
    # Expertise flags
    "expertise_auto",
    "expertise_home",
    "expertise_bundle",
    # Broker quality composite
    "broker_quality_score",
]
 
# ── Interaction features — added to BOTH towers' concat ────
# These bridge the two sides and help the loss surface converge faster.
# We'll handle them in a late-fusion head rather than in the towers.
INTERACTION_FEATURES = [
    "expertise_match",
    "language_match",
    "workload_ratio",
    "quality_x_value",
    "position_bias",
    "interaction_number"
]
 

Loaded: (20354, 116)
Conversion rate: 11.68%


/tmp/ipykernel_58386/754369054.py:1: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("prepared_positive_v81.csv")


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import numpy as np

In [4]:

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

EMBEDDING_DIM  = 64       # shared output dim of both towers
HIDDEN_DIM     = 128      # first hidden layer width
DROPOUT        = 0.3
LEARNING_RATE  = 1e-3
BATCH_SIZE     = 512
EPOCHS         = 30
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_IPS        = True     # weight loss by inverse propensity score
IPS_CLIP       = 10.0     # clip IPS weights to prevent variance explosion
 

In [5]:
def filter_cols(cols, df):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"  Warning — missing columns (skipped): {missing}")
    return [c for c in cols if c in df.columns]
 
CLIENT_FEATURES      = filter_cols(CLIENT_FEATURES, df)
BROKER_FEATURES      = filter_cols(BROKER_FEATURES, df)
INTERACTION_FEATURES = filter_cols(INTERACTION_FEATURES, df)
 
print(f"\nClient features:      {len(CLIENT_FEATURES)}")
print(f"Broker features:      {len(BROKER_FEATURES)}")
print(f"Interaction features: {len(INTERACTION_FEATURES)}")
 
all_features = CLIENT_FEATURES + BROKER_FEATURES + INTERACTION_FEATURES + ["converted", "propensity_score"]
df = df[all_features].fillna(0)
 

  Warning — missing columns (skipped): ['ribo_experience']

Client features:      20
Broker features:      16
Interaction features: 6


In [6]:
df_full = pd.read_csv("prepared_positive_v81.csv")
df_full = df_full.fillna(0)
 
# Reconstruct a sort key from temporal features
if "lead_year" in df_full.columns and "lead_quarter" in df_full.columns:
    df_full["_time_key"] = df_full["lead_year"] * 10 + df_full["lead_quarter"]
    df_full = df_full.sort_values("_time_key").reset_index(drop=True)
    n = len(df_full)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)
    train_df  = df_full.iloc[:train_end].copy()
    val_df    = df_full.iloc[train_end:val_end].copy()
    test_df   = df_full.iloc[val_end:].copy()
    print(f"Temporal split:")
else:
    print("Warning: no temporal features found — using random split")
    train_df, temp_df = train_test_split(df_full, test_size=0.30, random_state=SEED,
                                          stratify=df_full["converted"])
    val_df,  test_df  = train_test_split(temp_df,  test_size=0.50, random_state=SEED,
                                          stratify=temp_df["converted"])
    print(f"Random split:")
 
print(f"  Train: {len(train_df):,}  "
      f"({train_df['converted'].mean()*100:.1f}% positive)")
print(f"  Val:   {len(val_df):,}  "
      f"({val_df['converted'].mean()*100:.1f}% positive)")
print(f"  Test:  {len(test_df):,}  ")

Temporal split:
  Train: 14,247  (11.9% positive)
  Val:   3,053  (11.2% positive)
  Test:  3,054  


/tmp/ipykernel_58386/675459411.py:1: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv("prepared_positive_v81.csv")


In [7]:
print("=== Data Type Check ===")
for col in CLIENT_FEATURES:
    if col in df.columns:
        dtype = df[col].dtype
        if dtype == 'object':
            print(f"❌ {col}: {dtype}")
            print(f"   Unique values: {df[col].unique()[:5]}")
        else:
            print(f"✅ {col}: {dtype}")

=== Data Type Check ===
✅ quote_value_scaled: float64
✅ lead_difficulty_scaled: float64
✅ sophistication_scaled: float64
✅ patience_hours_scaled: float64
✅ digital_engagement_score_scaled: float64
✅ tenure_years_scaled: float64
✅ log_quote_value: float64
✅ log_patience_hours: float64
✅ month: float64
✅ hour_of_day: float64
✅ lead_dayofweek: float64
✅ lead_quarter: float64
❌ is_weekend: object
   Unique values: [False True]
✅ insurance_type_enc: int64
✅ claims_risk: int64
✅ multi_product_intent: int64
✅ insurance_type_missing: int64
✅ language_missing: int64
✅ tenure_years_missing: int64
✅ digital_engagement_score_missing: int64


In [8]:
def smart_convert_to_numeric(series, column_name):
    """Intelligently convert any series to numeric"""
    # If already numeric, return
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float)
    
    # Try direct conversion to numeric
    converted = pd.to_numeric(series, errors='coerce')
    
    # If conversion failed for many values, try boolean mapping
    if converted.isna().sum() > len(series) * 0.1:  # More than 10% failed
        # Get unique non-null values
        unique_vals = series.dropna().unique()
        
        if len(unique_vals) <= 5:  # Probably categorical
            print(f"  {column_name}: Converting categorical to numeric - values: {unique_vals[:5]}")
            # Create mapping dictionary
            mapping = {val: idx for idx, val in enumerate(sorted(unique_vals))}
            converted = series.map(mapping)
        else:
            print(f"  {column_name}: Complex object column - setting to 0")
            converted = pd.Series(0, index=series.index)
    
    return converted.fillna(0).astype(float)

class BrokerMatchDataset(Dataset):
    def __init__(self, df):
        # Smart conversion for all features
        self.client_x = self._create_tensor(df, CLIENT_FEATURES, "client")
        self.broker_x = self._create_tensor(df, BROKER_FEATURES, "broker")
        self.interaction_x = self._create_tensor(df, INTERACTION_FEATURES, "interaction")
        self.labels = self._create_tensor(df, ["converted"], "labels").squeeze()
        
        # IPS weights
        if "propensity_score" in df.columns:
            propensity = smart_convert_to_numeric(df["propensity_score"], "propensity_score")
            raw_ips = 1.0 / (propensity.clip(0.01, 1.0).values)
            self.ips_weights = torch.tensor(raw_ips.clip(max=IPS_CLIP), dtype=torch.float32)
        else:
            self.ips_weights = torch.ones(len(df), dtype=torch.float32)
    
    def _create_tensor(self, df, columns, name):
        """Safely create tensor from columns"""
        if not columns:
            return torch.tensor([], dtype=torch.float32)
        
        # Filter existing columns
        existing_cols = [c for c in columns if c in df.columns]
        missing = set(columns) - set(existing_cols)
        if missing:
            print(f"Warning: {name} - missing columns: {missing}")
        
        if not existing_cols:
            return torch.tensor([], dtype=torch.float32)
        
        # Convert each column
        df_subset = pd.DataFrame()
        for col in existing_cols:
            df_subset[col] = smart_convert_to_numeric(df[col], col)
        
        return torch.tensor(df_subset.values, dtype=torch.float32)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return (
            self.client_x[idx],
            self.broker_x[idx],
            self.interaction_x[idx],
            self.labels[idx],
            self.ips_weights[idx],
        )

# This will handle all conversions automatically
train_ds = BrokerMatchDataset(train_df)
val_ds = BrokerMatchDataset(val_df)
test_ds = BrokerMatchDataset(test_df)
 

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
 
print(f"\nDataLoaders ready:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")



DataLoaders ready:
  Train batches: 28
  Val batches:   6
  Test batches:  6


In [9]:
class Tower(nn.Module):
    """Single encoder tower: input_dim → HIDDEN_DIM → EMBEDDING_DIM, L2-normalised."""
 
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, embed_dim=EMBEDDING_DIM,
                 dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, embed_dim),
        )
 
    def forward(self, x):
        emb = self.net(x)
        # L2 normalise so dot product = cosine similarity ∈ [-1, 1]
        return nn.functional.normalize(emb, p=2, dim=1)
 
class TwoTowerModel(nn.Module):
    def __init__(self, client_dim, broker_dim, interaction_dim, embed_dim=EMBEDDING_DIM):
        super().__init__()
        self.client_tower = Tower(client_dim, embed_dim=embed_dim)
        self.broker_tower = Tower(broker_dim, embed_dim=embed_dim)

        # Include both embeddings + their elementwise product + dot + interactions
        fusion_input_dim = embed_dim + embed_dim + embed_dim + 1 + interaction_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT * 0.5),
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, client_x, broker_x, interaction_x):
        client_emb = self.client_tower(client_x)
        broker_emb = self.broker_tower(broker_x)
        dot = (client_emb * broker_emb).sum(dim=1, keepdim=True)
        elementwise = client_emb * broker_emb  # hadamard product
        fusion_input = torch.cat([client_emb, broker_emb, elementwise, dot, interaction_x], dim=1)
        return self.fusion_head(fusion_input).squeeze(1)
    
    def get_embeddings(self, client_x, broker_x):
        """Inference helper: returns (client_emb, broker_emb) for ANN lookup."""
        self.eval()
        with torch.no_grad():
            return self.client_tower(client_x), self.broker_tower(broker_x)
 
 
# Instantiate
model = TwoTowerModel(
    client_dim      = len(CLIENT_FEATURES),
    broker_dim      = len(BROKER_FEATURES),
    interaction_dim = len(INTERACTION_FEATURES),
).to(DEVICE)
 
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel architecture:")
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")


Model architecture:
TwoTowerModel(
  (client_tower): Tower(
    (net): Sequential(
      (0): Linear(in_features=20, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): Dropout(p=0.15, inplace=False)
      (8): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (broker_tower): Tower(
    (net): Sequential(
      (0): Linear(in_features=16, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [ ]:
pos_count = train_df["converted"].sum()
neg_count = len(train_df) - pos_count
pos_weight = torch.tensor([2 * neg_count / pos_count], dtype=torch.float32).to(DEVICE)
print(f"Class imbalance ratio (neg/pos): {neg_count/pos_count:.1f}x  →  pos_weight={pos_weight.item():.2f}")
 
bce_loss_fn = nn.BCEWithLogitsLoss(reduction="none") 
 
def weighted_bce(logits, labels, ips_weights):
    per_sample = bce_loss_fn(logits, labels)
    if USE_IPS:
        # Normalize so mean weight ≈ 1 — prevents loss explosion
        ips_weights = ips_weights / ips_weights.mean()
        per_sample = per_sample * ips_weights
    return per_sample.mean()
 
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
 
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3)
 

Class imbalance ratio (neg/pos): 7.4x  →  pos_weight=7.41


In [11]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, all_probs, all_labels = 0.0, [], []
 
    with torch.set_grad_enabled(train):
        for client_x, broker_x, inter_x, labels, ips_w in loader:
            client_x = client_x.to(DEVICE)
            broker_x = broker_x.to(DEVICE)
            inter_x  = inter_x.to(DEVICE)
            labels   = labels.to(DEVICE)
            ips_w    = ips_w.to(DEVICE)
 
            logits = model(client_x, broker_x, inter_x)
            loss   = weighted_bce(logits, labels, ips_w)
 
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
 
            total_loss += loss.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().detach().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    n = len(all_labels)
    avg_loss = total_loss / n
    auc      = roc_auc_score(all_labels, all_probs)
    pr_auc   = average_precision_score(all_labels, all_probs)
    return avg_loss, auc, pr_auc

In [12]:
best_val_auc  = 0.0
best_epoch    = 0
history       = []
 
print(f"\n{'Epoch':>5}  {'Train Loss':>10}  {'Train AUC':>10}  "
      f"{'Val Loss':>9}  {'Val AUC':>8}  {'Val PR-AUC':>10}")
print("-" * 60)
 
for epoch in range(1, EPOCHS + 1):
    train_loss, train_auc, train_pr = run_epoch(train_loader, train=True)
    val_loss,   val_auc,   val_pr   = run_epoch(val_loader,   train=False)
 
    scheduler.step(val_auc)
 
    history.append({
        "epoch": epoch,
        "train_loss": train_loss, "train_auc": train_auc, "train_pr": train_pr,
        "val_loss":   val_loss,   "val_auc":   val_auc,   "val_pr":   val_pr,
    })
 
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch   = epoch
        torch.save(model.state_dict(), "two_tower_best.pt")
 
    print(f"{epoch:>5}  {train_loss:>10.4f}  {train_auc:>10.4f}  "
          f"{val_loss:>9.4f}  {val_auc:>8.4f}  {val_pr:>10.4f}"
          + ("  ← best" if epoch == best_epoch else ""))
 
print(f"\nBest model: epoch {best_epoch}  |  val AUC = {best_val_auc:.4f}")


Epoch  Train Loss   Train AUC   Val Loss   Val AUC  Val PR-AUC
------------------------------------------------------------
    1      0.5921      0.4899     0.3667    0.4716      0.1035  ← best
    2      0.3743      0.4863     0.3497    0.5193      0.1156  ← best
    3      0.3657      0.5355     0.3469    0.5679      0.1387  ← best
    4      0.3594      0.5973     0.3454    0.5832      0.1431  ← best
    5      0.3561      0.6190     0.3467    0.5927      0.1441  ← best
    6      0.3530      0.6385     0.3425    0.6136      0.1539  ← best
    7      0.3529      0.6389     0.3416    0.6196      0.1556  ← best
    8      0.3480      0.6636     0.3409    0.6291      0.1586  ← best
    9      0.3470      0.6674     0.3396    0.6344      0.1611  ← best
   10      0.3453      0.6722     0.3389    0.6394      0.1626  ← best
   11      0.3433      0.6828     0.3372    0.6475      0.1672  ← best
   12      0.3422      0.6862     0.3398    0.6489      0.1683  ← best
   13      0.3413      

In [13]:
model.load_state_dict(torch.load("two_tower_best.pt", map_location=DEVICE))
model.eval()
 
test_loss, test_auc, test_pr_auc = run_epoch(test_loader, train=False)
 
# Threshold at 0.5 for classification report
all_probs, all_labels = [], []
with torch.no_grad():
    for client_x, broker_x, inter_x, labels, _ in test_loader:
        logits = model(client_x.to(DEVICE), broker_x.to(DEVICE), inter_x.to(DEVICE))
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labels.numpy())
 
all_preds = (np.array(all_probs) >= 0.5).astype(int)
 
print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  Loss:    {test_loss:.4f}")
print(f"  AUC-ROC: {test_auc:.4f}")
print(f"  PR-AUC:  {test_pr_auc:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=["no convert", "convert"]))

TEST SET RESULTS
  Loss:    0.3292
  AUC-ROC: 0.6830
  PR-AUC:  0.1940

              precision    recall  f1-score   support

  no convert       0.89      1.00      0.94      2713
     convert       0.00      0.00      0.00       341

    accuracy                           0.89      3054
   macro avg       0.44      0.50      0.47      3054
weighted avg       0.79      0.89      0.84      3054



/home/lang-chain/Documents/Agentic_AI_MCP_RAG_Chat/.conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lang-chain/Documents/Agentic_AI_MCP_RAG_Chat/.conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lang-chain/Documents/Agentic_AI_MCP_RAG_Chat/.conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter

In [14]:
print(f"pos_weight: {pos_weight.item():.2f}")  # Should be ~8-12x
print(f"Train positives: {train_df['converted'].sum()} / {len(train_df)}")

pos_weight: 7.41
Train positives: 1695 / 14247
